In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc PyGithub networkx qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*Lihat [rujukan API](https://docs.quantum.ibm.com/api/functions/kipu-optimization)*

> **Note:** * Fungsi Qiskit ialah ciri eksperimental yang hanya tersedia untuk pengguna Pelan Premium IBM Quantum&reg;, Pelan Flex, dan Pelan On-Prem (melalui IBM Quantum Platform API). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.

## Gambaran keseluruhan

Dengan Iskay Quantum Optimizer oleh Kipu Quantum, anda boleh menangani masalah pengoptimuman yang rumit menggunakan komputer kuantum IBM&reg;. Penyelesai ini memanfaatkan [algoritma bf-DCQO](https://doi.org/10.48550/arXiv.2409.04477) mutakhir Kipu yang hanya memerlukan fungsi objektif sebagai input untuk menghasilkan penyelesaian masalah secara automatik. Ia boleh mengendalikan masalah pengoptimuman yang melibatkan sehingga 156 qubit, membolehkan penggunaan semua qubit pada peranti kuantum IBM. Pengoptimum menggunakan pemetaan 1-ke-1 antara pemboleh ubah klasik dan qubit, yang membolehkan anda menangani masalah pengoptimuman dengan sehingga 156 pemboleh ubah binari.

Pengoptimum ini membolehkan penyelesaian masalah pengoptimuman binari tanpa kekangan. Selain formulasi QUBO (Quadratic Unconstrained Binary Optimization) yang lazim digunakan, ia juga menyokong masalah pengoptimuman peringkat tinggi (HUBO). Penyelesai ini menggunakan algoritma kuantum bukan variasi, yang menjalankan sebahagian besar pengiraan pada peranti kuantum.

Berikut memberikan lebih butiran tentang algoritma yang digunakan dan panduan ringkas tentang cara menggunakan fungsi ini, serta keputusan penanda aras pada pelbagai contoh masalah dengan saiz dan kerumitan yang berbeza.

## Penerangan

Pengoptimum ini ialah pelaksanaan sedia guna bagi algoritma pengoptimuman kuantum mutakhir. Ia menyelesaikan masalah pengoptimuman dengan menjalankan litar kuantum yang dimampatkan dengan ketara pada perkakasan kuantum. Pemampatan ini dicapai dengan memperkenalkan sebutan counterdiabatic ke dalam evolusi masa asas sistem kuantum. Algoritma menjalankan beberapa lelaran proses perkakasan untuk memperoleh penyelesaian akhir dan menggabungkannya dengan pasca-pemprosesan. Langkah-langkah ini disepadukan dengan lancar ke dalam alur kerja Pengoptimum dan dilaksanakan secara automatik.

### Bagaimana Pengoptimum Kuantum berfungsi?

Bahagian ini menggariskan asas algoritma bf-DCQO yang dilaksanakan. Pengenalan kepada algoritma ini juga boleh didapati di [saluran YouTube Qiskit](https://www.youtube.com/watch?v=33QmsXhIlpU&t=1223s).

Algoritma ini berasaskan evolusi masa sistem kuantum yang berubah dari semasa ke semasa, di mana penyelesaian masalah dikodkan dalam keadaan asas sistem kuantum pada penghujung evolusi. Menurut [teorem adiabatik](https://en.wikipedia.org/wiki/Adiabatic_theorem), evolusi ini mesti berlaku perlahan untuk memastikan sistem kekal dalam keadaan asasnya. Mendigitkan evolusi ini adalah asas kepada pengiraan adiabatik kuantum terdigit (DQA) dan algoritma QAOA yang terkenal. Walau bagaimanapun, evolusi perlahan yang diperlukan tidak praktikal untuk saiz masalah yang semakin besar kerana ia menghasilkan kedalaman litar yang semakin meningkat. Dengan menggunakan protokol counterdiabatic, anda boleh menindas pengujaan yang tidak diingini semasa masa evolusi yang singkat sambil kekal dalam keadaan asas. Di sini, mendigitkan masa evolusi yang lebih singkat ini menghasilkan litar kuantum dengan kedalaman yang lebih pendek dan bilangan get yang mengaitkan yang lebih sedikit.

Litar algoritma bf-DCQO biasanya menggunakan sehingga sepuluh kali lebih sedikit get pengkaitan berbanding DQA, dan tiga hingga empat kali lebih sedikit get pengkaitan berbanding pelaksanaan QAOA standard. Disebabkan bilangan get yang lebih kecil, lebih sedikit ralat berlaku semasa pelaksanaan litar pada perkakasan. Oleh itu, pengoptimum ini tidak memerlukan penggunaan teknik seperti penindasan ralat atau pengurangan ralat. Melaksanakannya dalam versi masa depan boleh meningkatkan kualiti penyelesaian dengan lebih lanjut.

Walaupun algoritma bf-DCQO menggunakan lelaran, ia bukan variasi. Selepas setiap lelaran algoritma, taburan keadaan diukur. Taburan yang diperoleh digunakan untuk mengira medan berat sebelah (bias-field). Medan berat sebelah membolehkan permulaan lelaran seterusnya dari keadaan tenaga yang hampir dengan penyelesaian yang ditemui sebelumnya. Dengan cara ini, algoritma bergerak dengan setiap lelaran ke arah penyelesaian dengan tenaga yang lebih rendah. Biasanya, kira-kira sepuluh lelaran sudah mencukupi untuk menumpu kepada penyelesaian, dengan jumlah bilangan lelaran yang jauh lebih rendah berbanding algoritma variasi, iaitu dalam susunan kira-kira 100 lelaran.

Pengoptimum menggabungkan algoritma bf-DCQO dengan pasca-pemprosesan klasik. Selepas mengukur taburan keadaan, carian tempatan dilakukan. Semasa carian tempatan, bit penyelesaian yang diukur dibalikkan secara rawak. Selepas pembalikan, tenaga bitstring baru dinilai. Jika tenaganya lebih rendah, bitstring itu dikekalkan sebagai penyelesaian baru. Carian tempatan hanya berskala secara linear dengan bilangan qubit; oleh itu, ia murah dari segi pengiraan. Oleh kerana pasca-pemprosesan membetulkan pembalikan bit setempat, ia mengimbangi ralat pembalikan bit yang sering menjadi hasil ketidaksempurnaan perkakasan dan ralat pembacaan.

### Alur kerja

Berikut ialah skema alur kerja Pengoptimum Kuantum.

![Alur kerja](../docs/images/guides/kipu-optimization/workflow.svg "Alur kerja Pengoptimum Kuantum")

Dengan menggunakan Pengoptimum Kuantum, menyelesaikan masalah pengoptimuman pada perkakasan kuantum boleh dikurangkan kepada:
* Merumuskan fungsi objektif masalah
* Mengakses Pengoptimum melalui Fungsi Qiskit
* Menjalankan Pengoptimum dan mengumpul keputusan

## Penanda aras

Metrik penanda aras di bawah menunjukkan bahawa Pengoptimum menangani masalah yang melibatkan sehingga 156 qubit dengan berkesan dan memberikan gambaran keseluruhan ketepatan dan kebolehskalaan pengoptimum merentasi pelbagai jenis masalah. Perhatikan bahawa metrik prestasi sebenar mungkin berbeza bergantung pada ciri masalah tertentu, seperti bilangan pemboleh ubah, ketumpatan dan lokaliti sebutan dalam fungsi objektif, dan peringkat polinomial.

Jadual berikut merangkumi nisbah penghampiran (AR), iaitu metrik yang ditakrifkan seperti berikut:
$$
AR = \frac{C^{*} - C_\textrm{max}}{C_{\textrm{min}} - C_{\textrm{max}}},
$$
di mana $C$ ialah fungsi objektif, $C_{\textrm{min}}$, $C_{\textrm{max}}$ ialah nilai minimum dan maksimumnya, dan $C^{*}$ ialah kos penyelesaian terbaik yang ditemui. Oleh itu, AR=100\% bermakna keadaan asas masalah telah diperoleh.

| Contoh            | Bilangan qubit | Nisbah Penghampiran | Jumlah masa (s) | Penggunaan Runtime (s) | Jumlah bilangan shot | Bilangan lelaran |
| ----------------- | :--------------: | :------: | :------------: | :---------------: | :-------------------: | :------------------: |
| MaxCut Tidak Berwajaran |        28        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut Tidak Berwajaran |        30        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut Tidak Berwajaran |        32        |   100%   |      180       |        30         |          30k          |          5           |
| MaxCut Tidak Berwajaran |        80        |   100%   |      480       |        60         |          90k          |          9           |
| MaxCut Tidak Berwajaran |       100        |   100%   |      330       |        60         |          60k          |          6           |
| MaxCut Tidak Berwajaran |       120        |   100%   |      370       |        60         |          60k          |          6           |
| HUBO 1            |       156        |   100%   |      600       |        70         |         100k          |          10          |
| HUBO 2            |       156        |   100%   |      600       |        70         |         100k          |          10          |

- Contoh MaxCut dengan 28, 30, dan 32 qubit telah dijalankan pada ibm_sherbrooke. Contoh dengan 80, 100, dan 120 dijalankan pada pemproses Heron r2.
- Contoh HUBO juga dijalankan pada pemproses Heron r2.

Semua contoh penanda aras boleh diakses di GitHub (lihat [contoh penanda aras Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances)). Contoh untuk menjalankan contoh ini boleh didapati dalam [Contoh 3: Contoh penanda aras](#example-3-benchmark-instances).

## Mula

Dalam dokumentasi ini, kita akan melalui langkah-langkah menggunakan Iskay Quantum Optimizer. Dalam proses ini kita akan menunjukkan secara ringkas cara memuatkan fungsi daripada katalog dan cara menukar masalah anda kepada input yang sah, sambil menunjukkan cara anda boleh bereksperimen dengan parameter pilihan yang berbeza.

Untuk contoh yang lebih terperinci, semak tutorial [Selesaikan masalah Market Split dengan Iskay Quantum Optimizer Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer), di mana kita melalui keseluruhan proses menggunakan Iskay Solver untuk menangani masalah Market Split, yang mewakili cabaran peruntukan sumber dunia sebenar di mana pasaran mesti dibahagikan kepada wilayah jualan yang seimbang untuk memenuhi sasaran permintaan tepat.

Sahkan menggunakan kunci API anda, yang boleh didapati pada [papan pemuka IBM Quantum Platform](http://quantum.cloud.ibm.com/), dan pilih Fungsi Qiskit seperti berikut:

In [ ]:
# ruff: noqa: F821

> **Note:** Kod berikut mengandaikan bahawa anda telah menyimpan bukti kelayakan anda. Jika belum, ikuti arahan dalam [simpan akaun IBM Cloud anda](/guides/functions#install-qiskit-functions-catalog-client) untuk mengesahkan dengan kunci API anda.

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
)

# Access Function
optimizer = catalog.load("kipu-quantum/iskay-quantum-optimizer")

## Contoh konfigurasi tersuai
Berikut adalah cara anda boleh mengkonfigurasi Iskay dengan tetapan yang berbeza:

In [ ]:
custom_options = {
    "shots": 15_000,  # Higher shot count for better statistics
    "num_iterations": 12,  # More iterations for solution refinement
    "preprocessing_level": 1,  # Light preprocessing for problem simplification
    "postprocessing_level": 2,  # Maximum postprocessing for solution quality
    "transpilation_level": 3,  # Using higher transpilation level for circuit optimization
    "seed_transpiler": 42,  # Fixed seed for reproducible results
    "job_tags": ["custom_config"],  # Custom tracking tags
}

**Pengoptimuman benih**: Perhatikan bahawa `seed_transpiler` ditetapkan kepada `None` secara lalai. Ini membolehkan proses pengoptimuman automatik Transpiler. Apabila `None`, sistem akan memulakan percubaan dengan beberapa benih dan memilih yang menghasilkan kedalaman Circuit terbaik, memanfaatkan sepenuhnya parameter `max_trials` untuk setiap tahap transpilasi.

**Prestasi tahap transpilasi**: Meningkatkan bilangan `max_trials` dengan nilai yang lebih tinggi untuk `transpilation_level` tidak dapat tidak akan meningkatkan masa transpilasi, tetapi ia mungkin tidak sentiasa mengubah litar akhir — ini sangat bergantung pada struktur dan kerumitan Circuit tertentu. Walau bagaimanapun, untuk sesetengah Circuit/masalah, perbezaan antara 10 percubaan (tahap 1) dan 50 percubaan (tahap 5) boleh menjadi dramatik, jadi meneroka parameter ini mungkin menjadi kunci untuk berjaya mencari penyelesaian.

## Contoh 1: Fungsi kos mudah

Pertimbangkan fungsi kos dalam formulasi spin:
$$
C(x_0, x_1, x_2, x_3, x_4) = 1 + 1.5x_0 + 2x_1 + 1.3x_2 + 2.5x_0x_3 + 3.5x_1x_4 + 4x_0x_1x_2
$$

di mana $(x_0, ..., x_4) \in {-1, 1}^5$.

Penyelesaian kepada fungsi kos mudah ini ialah
$$
(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)
$$
dengan nilai minimum $C^{*} = -6$

### 1. Cipta fungsi objektif

Kita mulakan dengan mencipta kamus dengan pekali fungsi objektif seperti berikut:

In [ ]:
objective_func = {
    "()": 1,
    "(0,)": 1.5,
    "(1,)": 2,
    "(2,)": 1.3,
    "(0, 3)": 2.5,
    "(1, 4)": 3.5,
    "(0, 1, 2)": 4,
}

### 2. Jalankan Pengoptimum
Kita selesaikan masalah dengan menjalankan pengoptimum. Oleh kerana $(x_0, ..., x_4) \in {-1, 1}^5$ kita mesti menetapkan `problem_type=spin`.

In [ ]:
# Setup options to run the optimizer
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Dapatkan keputusan
Penyelesaian masalah pengoptimuman diberikan terus daripada pengoptimum.

In [ ]:
print(job.result())

Ini akan memaparkan kamus dalam bentuk:

```
{'solution': {'0': -1, '1': -1, '2': -1, '3': 1, '4': 1},
 'solution_info': {'bitstring': '11100',
  'cost': -13.8,
  'seed_transpiler': 42,
  'mapping': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}},
 'prob_type': 'spin'}
```

Perhatikan bahawa kamus `solution` memaparkan vektor keputusan $(x_0, x_1, x_2, x_3, x_4) = (-1, -1, -1, 1, 1)$.

## Contoh 2: MaxCut

Banyak masalah graf seperti MaxCut atau set bebas maksimum ialah masalah NP-susah dan calon ideal untuk menguji algoritma kuantum dan perkakasan. Contoh ini menunjukkan penyelesaian masalah MaxCut bagi graf 3-sekata dengan Pengoptimum Kuantum.

Untuk menjalankan contoh ini anda mesti memasang pakej `networkx` selain daripada `qiskit-ibm-catalog`. Untuk memasangnya, jalankan arahan berikut:

In [ ]:
# %pip install networkx numpy

### 1. Cipta fungsi objektif
Mulakan dengan menjana graf 3-sekata rawak. Untuk graf ini, kita takrifkan fungsi objektif masalah MaxCut.

In [ ]:
import networkx as nx

# Create a random 3-regular graph
G = nx.random_regular_graph(3, 10, seed=42)


# Create the objective function for MaxCut in Ising formulation
def graph_to_ising_maxcut(G):
    """
    Convert a NetworkX graph to an Ising Hamiltonian for the max-cut problem.
    Args:
        G (networkx.Graph): The input graph.
    Returns:
        dict: The objective function of the Ising model
    """
    # Initialize the linear and quadratic coefficients
    objective_func = {}
    # Populate the coefficients
    for i, j in G.edges:
        objective_func[f"({i}, {j})"] = 0.5
    return objective_func


objective_func = graph_to_ising_maxcut(G)

### 2. Jalankan Pengoptimum
Selesaikan masalah dengan menjalankan pengoptimum.

In [ ]:
options = {"shots": 5000, "num_iterations": 5, "use_session": True}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": backend_name,  # such as "ibm_fez"
    "options": options,
}

job = optimizer.run(**arguments)

### 3. Dapatkan keputusan
Dapatkan keputusan dan petakan semula bitstring penyelesaian ke nod graf asal.

In [ ]:
print(job.result())

Penyelesaian kepada masalah Maxcut terkandung terus dalam sub-kamus `solution` objek keputusan

In [ ]:
maxcut_solution = job.result()["solution"]

## Contoh 3: Contoh penanda aras
Contoh penanda aras tersedia di GitHub: [Contoh penanda aras Kipu](https://github.com/Kipu-Quantum-GmbH/benchmark-instances).

Contoh boleh dimuatkan menggunakan pustaka `pygithub`. Untuk memasangnya, jalankan arahan berikut:

In [ ]:
# %pip install pygithub

Laluan untuk contoh penanda aras ialah:

**Maxcut:**
- `'maxcut/maxcut_28_nodes.json'`
- `'maxcut/maxcut_30_nodes.json'`
- `'maxcut/maxcut_32_nodes.json'`
- `'maxcut/maxcut_80_nodes.json'`
- `'maxcut/maxcut_100_nodes.json'`
- `'maxcut/maxcut_120_nodes.json'`

**HUBO:**
- `'HUBO/hubo1_marrakesh.json'`
- `'HUBO/hubo2_marrakesh.json'`

Untuk menghasilkan semula prestasi penanda aras bagi contoh HUBO, pilih Backend `ibm_marrakesh` dan tetapkan `direct_qubit_mapping` kepada `True` dalam sub-kamus `options`.

Contoh berikut menjalankan contoh Maxcut dengan 32 nod.

In [ ]:
from github import Github
import urllib
import json
import ast

repo = "Kipu-Quantum-GmbH/benchmark-instances"
path = "maxcut/maxcut_regular_3_150_nodes_weighted.json"
gh = Github()
repo = gh.get_repo(repo)
branch = "main"
file = repo.get_contents(urllib.parse.quote(path), ref=branch)

# load json file with benchmark problem
problem_json = json.loads(file.decoded_content)

# convert objective function to compatible format
objective_func = {
    key: ast.literal_eval(value) for key, value in problem_json.items()
}


# Setup configuration to run the optimizer
options = {
    "shots": 5_000,
    "num_iterations": 5,
    "use_session": True,
    "direct_qubit_mapping": False,
}

arguments = {
    "problem": objective_func,
    "problem_type": "spin",
    "backend_name": "<BACKEND-NAME>",
    "options": options,
}

job = optimizer.run(**arguments)

result = job.result()

## Kes penggunaan
Kes penggunaan tipikal untuk penyelesai Pengoptimuman ialah masalah pengoptimuman kombinatori. Anda boleh menyelesaikan masalah daripada banyak industri seperti kewangan, farmaseutik, atau logistik. Beberapa contoh berikut:
* Pengoptimuman portfolio (QUBO): [penerbitan saintifik](https://doi.org/10.1103/PhysRevApplied.22.054037) dan [kertas putih](https://kipu-quantum.com/zope64/kipu_2024/content/e3915/e3916/e4187/White-Paper-2-Financial-modeling-on-quantum-computers-using-digitally-compressed-algorithms-1.pdf)
* Pelipatan protein (HUBO): [penerbitan saintifik](https://doi.org/10.1103/PhysRevApplied.20.014024)
* Penjadualan logistik (QUBO): [penerbitan saintifik](https://doi.org/10.1103/PhysRevApplied.22.064068)
* Pengoptimuman rangkaian: [webinar](https://www.youtube.com/watch?v=w5SrCIK88No)
* Pembahagian pasaran (QUBO): [tutorial](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer)

Jika anda berminat untuk menangani kes penggunaan tertentu dan membangunkan pemetaan khusus, kami boleh membantu anda. [Hubungi kami](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).

## Dapatkan sokongan
Untuk sokongan, hubungi support@kipu-quantum.com.

## Langkah seterusnya
- [Minta akses kepada Pengoptimum Kuantum oleh Kipu Quantum](https://share-eu1.hsforms.com/2Ff8cgWvTR9ukT_fPoaNhDw2dqpz5).
- Lawati [rujukan API](https://docs.quantum.ibm.com/api/functions/kipu-optimization) untuk Fungsi Qiskit ini.
- Cuba tutorial [Selesaikan masalah Market Split dengan Iskay Quantum Optimizer Kipu Quantum](/tutorials/solve-market-split-problem-with-iskay-quantum-optimizer).
- Semak [Romero, S. V., et al. (2025).  Bias-Field Digitized Counterdiabatic Quantum Algorithm for Higher-Order Binary Optimization. arXiv preprint arXiv:2409.04477](https://arxiv.org/abs/2409.04477).
- Semak [Cadavid, A. G., et al. (2024).  Bias-field digitized counterdiabatic quantum optimization. arXiv preprint arXiv:2405.13898](https://arxiv.org/abs/2405.13898).
- Semak [Chandarana, P., et al. (2025).  Runtime Quantum Advantage with Digital Quantum Optimization. arXiv preprint arXiv:2505.08663](https://arxiv.org/abs/2505.08663).

## Maklumat tambahan
Iskay, seperti nama syarikat kami Kipu Quantum, ialah perkataan Peru. Walaupun kami ialah syarikat permulaan dari Jerman, perkataan-perkataan ini berasal dari negara asal salah seorang pengasas bersama kami, di mana Quipu ialah salah satu mesin pengiraan pertama yang dibangunkan oleh manusia 2000 tahun SM.